## 1. Load and Import

In [25]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
from shapely.ops import nearest_points
import warnings
warnings.filterwarnings("ignore", message="Could not parse column 'reversed'")

DATA_PROC = Path("../data/processed")
DATA_EXT  = Path("../data/external")
DATA_OUT  = Path("../data/processed")

In [26]:
tracts    = gpd.read_file(DATA_PROC / "svi_tracts.geojson")
stations  = gpd.read_file(DATA_PROC / "all_stations.geojson")
hydrants  = gpd.read_file(DATA_PROC / "Fire_Hydrants.geojson")
hospitals = gpd.read_file(DATA_PROC / "hospitals.geojson")
roads     = gpd.read_file(DATA_EXT  / "escape_routes_roads.geojson")

# join key: "tract" (float64, e.g. 1011.1) — full FIPS was dropped during SVI processing
# RPL columns are snake_case: rpl_themes, rpl_theme_1..4
# 25 tracts have rpl_themes == NaN (originally -999 sentinels, replaced at processing time)

CRS = "EPSG:3310"  # CA Albers, meters — best for LA distance/area calculations
tracts    = tracts.to_crs(CRS)
stations  = stations.to_crs(CRS)
hydrants  = hydrants.to_crs(CRS)
hospitals = hospitals.to_crs(CRS)
roads     = roads.to_crs(CRS)

print(tracts.shape, stations.shape, hydrants.shape, hospitals.shape)
print("rpl_themes NaNs:", tracts["rpl_themes"].isna().sum())

(2493, 152) (412, 6) (60295, 3) (165, 4)
rpl_themes NaNs: 25


## Feature 1: Distance to nearest fire station (per tract centroid)

In [27]:
tract_centroids = tracts.copy()
tract_centroids["geometry"] = tracts.geometry.centroid

def nearest_distance(origins, destinations):
    """Returns array of distances (meters) from each origin to nearest destination."""
    dest_union = destinations.geometry.union_all()
    return origins.geometry.apply(
        lambda pt: pt.distance(nearest_points(pt, dest_union)[1])
    )

tracts["dist_fire_station_m"] = nearest_distance(tract_centroids, stations)
print(tracts["dist_fire_station_m"].describe())

count     2493.000000
mean      1337.833240
std       1747.598043
min         21.555872
25%        791.078160
50%       1177.750111
75%       1607.861827
max      62357.005818
Name: dist_fire_station_m, dtype: float64


## Feature 2: Distance to nearest hospital

In [28]:
tracts["dist_hospital_m"] = nearest_distance(tract_centroids, hospitals)
print(tracts["dist_hospital_m"].describe())

count     2493.000000
mean      2983.523795
std       2877.505335
min         61.365995
25%       1413.948820
50%       2401.161141
75%       3672.355071
max      37682.365498
Name: dist_hospital_m, dtype: float64


##  Feature 3: Hydrant density (hydrants per km² within tract)

In [29]:
hydrants_in_tracts = gpd.sjoin(
    hydrants,
    tracts[["tract", "geometry"]],
    how="inner",
    predicate="within"
)
hydrant_counts = hydrants_in_tracts.groupby("tract").size().rename("hydrant_count")

if "hydrant_count" in tracts.columns:
    tracts = tracts.drop(columns=["hydrant_count"])
tracts = tracts.merge(hydrant_counts, on="tract", how="left")
tracts["hydrant_count"] = tracts["hydrant_count"].fillna(0)
tracts["tract_area_km2"] = tracts.geometry.area / 1e6
tracts["hydrant_density"] = tracts["hydrant_count"] / tracts["tract_area_km2"]

print(tracts["hydrant_density"].describe())

count    2493.000000
mean       30.952924
std        38.187512
min         0.000000
25%         0.000000
50%         0.000000
75%        64.421846
max       261.355381
Name: hydrant_density, dtype: float64


## Road escape density (road length per km² within tract)


In [30]:
roads_clipped = gpd.overlay(
    roads.to_crs("EPSG:3310"),
    tracts[["tract", "geometry"]].to_crs("EPSG:3310"),
    how="intersection"
)
roads_clipped["length_m"] = roads_clipped.geometry.length

road_length_per_tract = (
    roads_clipped.groupby("tract")["length_m"]
    .sum()
    .rename("road_length_m")
)

tracts = tracts.to_crs("EPSG:3310")
tracts["tract_area_km2"] = tracts.geometry.area / 1e6
if "road_length_m" in tracts.columns:
    tracts = tracts.drop(columns=["road_length_m"])
tracts = tracts.merge(road_length_per_tract, on="tract", how="left")
tracts["road_length_m"] = tracts["road_length_m"].fillna(0)
tracts["road_density"] = tracts["road_length_m"] / tracts["tract_area_km2"]
print(tracts["road_density"].describe())

count     2493.000000
mean      7171.994281
std       4150.211696
min          0.000000
25%       4309.327072
50%       6683.852786
75%       9372.396721
max      32131.626521
Name: road_density, dtype: float64


## Feature 5: SVI vulnerability scores

CDC SVI percentile scores (0–1, higher = more vulnerable).  
Column names are snake_case in the processed file: `rpl_themes`, `rpl_theme_1`–`rpl_theme_4`.  
25 tracts have NaN values (original -999 sentinels replaced during preprocessing) — imputed with the column median.

In [31]:
svi_cols = ["rpl_themes", "rpl_theme_1", "rpl_theme_2", "rpl_theme_3", "rpl_theme_4"]

print("NaN counts before imputation:")
print(tracts[svi_cols].isna().sum())

for col in svi_cols:
    tracts[col] = tracts[col].fillna(tracts[col].median())

print("\nNaN counts after imputation:")
print(tracts[svi_cols].isna().sum())
print()
print(tracts[svi_cols].describe())

NaN counts before imputation:
rpl_themes     25
rpl_theme_1    23
rpl_theme_2    21
rpl_theme_3    17
rpl_theme_4    24
dtype: int64

NaN counts after imputation:
rpl_themes     0
rpl_theme_1    0
rpl_theme_2    0
rpl_theme_3    0
rpl_theme_4    0
dtype: int64

        rpl_themes  rpl_theme_1  rpl_theme_2  rpl_theme_3  rpl_theme_4
count  2493.000000  2493.000000  2493.000000  2493.000000  2493.000000
mean      0.582855     0.591596     0.519223     0.632620     0.552221
std       0.286315     0.281469     0.286171     0.292414     0.283518
min       0.000600     0.000700     0.000000     0.001500     0.000000
25%       0.350200     0.358900     0.268500     0.392600     0.318900
50%       0.632850     0.633300     0.552950     0.708700     0.578600
75%       0.830300     0.841000     0.757900     0.897000     0.796600
max       1.000000     0.999900     0.999400     0.994800     1.000000


## Save feature matrix

In [32]:
feature_cols = [
    "tract",
    "dist_fire_station_m",
    "dist_hospital_m",
    "hydrant_density",
    "road_density",
    "rpl_themes",
    "rpl_theme_1",
    "rpl_theme_2",
    "rpl_theme_3",
    "rpl_theme_4",
    "geometry",
]

features = tracts[feature_cols].copy()
print(features.shape)
print(features.isna().sum())
features.to_file(DATA_OUT / "features.geojson", driver="GeoJSON")
print("Saved to", DATA_OUT / "features.geojson")

(2493, 11)
tract                  0
dist_fire_station_m    0
dist_hospital_m        0
hydrant_density        0
road_density           0
rpl_themes             0
rpl_theme_1            0
rpl_theme_2            0
rpl_theme_3            0
rpl_theme_4            0
geometry               0
dtype: int64
Saved to ../data/processed/features.geojson
